In [2]:
import os
from dotenv import load_dotenv
import json
import pathlib
import textwrap
import google.generativeai as genai
from IPython.display import display
from IPython.display import Markdown
from google.api_core import retry
import typing_extensions as typing

In [3]:

def to_markdown(text):
    text = text.replace('•', '  *')
    return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [4]:
load_dotenv()
google_api_key = os.getenv('GEMINI_API_KEY')
genai.configure(api_key=google_api_key)

In [5]:
system_prompt ={"text":"you are an medical AI agent.Respond the reaction between the drugs to human when taken together in 100 words"}


In [6]:
# tool to add medicine names to a list
medicine_list = []
def medicine_tool(medicine):
    med = medicine.lower().strip()
    if med not in medicine_list:
        medicine_list.append(med)
    return medicine_list
    

In [7]:
medicine_class={
    "name": "medicine_tool",
    "description":"gets the list of medicines",
    "parameters":{
        "type":"object",
        "properties":{
            "medicine":{
                "type":"string",
                "description":"The medicine name",
            },
        },
        "required":["medicine"],
        "additionalProperties":False
    }
}

In [8]:
tool_config = {
    "function_calling_config": {
        "mode": "ANY",
        "allowed_function_names": ["medicine_class"]
    }
}


In [9]:
user_prompt=f"what is the reaction between the drugs in the list:\n"
user_prompt+=str([medicine_class])

In [10]:
def message_gemini(*user_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    gemini = genai.GenerativeModel(
    model_name='gemini-1.5-flash',
    system_instruction=system_prompt,
    tool_config=tool_config
)
    response = gemini.generate_content(user_prompt)
    return response.text

In [12]:
to_markdown(message_gemini("Rosuvastatin-H 10mg","Losartan 25mg","bisoprolol Fumarate 7.5mg","Silodosin 8mg","Feburic 40mg"))

> Combining rosuvastatin, losartan, bisoprolol, silodosin, and febuxostat can lead to several potential interactions.  Rosuvastatin's muscle pain risk is increased by febuxostat.  Bisoprolol and losartan together may lower blood pressure excessively, increasing fainting risk.  Silodosin, an alpha-blocker, may increase the hypotensive effects of bisoprolol and losartan.  Febuxostat may also increase bleeding risk if already taking medications that affect blood clotting.  This combination necessitates close medical monitoring for hypotension, muscle pain (myalgia/rhabdomyolysis), and bleeding.  Always consult your doctor before taking multiple medications simultaneously.
